In [ ]:
import feedparser

RSS_FEEDS = [
    "https://news.google.com/rss/search?q=artificial+intelligence+LLM&hl=en&gl=US&ceid=US:en",
    "https://hnrss.org/newest?q=LLM+AI+language+model",
]

print("=== TESTING NEWS SOURCES ===\n")
for url in RSS_FEEDS:
    feed = feedparser.parse(url)
    print(f"Source: {feed.feed.get('title', url)}")
    print(f"Articles fetched: {len(feed.entries)}")
    for entry in feed.entries[:3]:
        print(f"  - {entry.get('title', 'no title')}")
    print()

In [ ]:
# !pip install feedparser

In [ ]:
# test_ner_local.py — run from pipeline/ folder
from transformers import pipeline

print("Loading model (first run downloads ~400MB)...")
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("Model loaded.\n")

# Mix of known + new entities — this is exactly the drift test
titles = [
    "OpenAI releases GPT-5 to compete with Google Gemini",      # known — should be high confidence
    "DeepSeek R2 outperforms all models on reasoning benchmarks", # OOV — should be low confidence
    "Anthropic raises funding led by Amazon",                    # known
    "SSI founded by Ilya Sutskever announces first model",       # OOV
    "Grok 3 from xAI beats Claude on coding tasks",             # OOV
    "Meta releases Llama 4 Scout as open source",               # OOV
]

print("=== NER INFERENCE TEST ===\n")
for title in titles:
    results = ner(title)
    print(f"Title: {title}")
    for r in results:
        flag = "⚠️  LOW" if r['score'] < 0.70 else "✅ OK "
        print(f"  {flag}  '{r['word']}'  ({r['entity_group']})  conf={r['score']:.3f}")
    print()

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
for word in ["DeepSeek", "Grok", "SSI", "Llama", "xAI"]:
    tokens = tokenizer.tokenize(word)
    print(f"{word}: {tokens}")

In [ ]:
# test_ner_on_real_news.py
import feedparser
from transformers import pipeline

ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

feed = feedparser.parse(
    "https://news.google.com/rss/search?q=artificial+intelligence+LLM&hl=en&gl=US&ceid=US:en"
)

def is_valid_span(word):
    if word.startswith("##"):           return False
    if len(word.strip("#")) <= 1:       return False
    if word.isupper() and len(word) <= 2: return False
    return True

all_confidences = []
flagged = []

for entry in feed.entries[:30]:
    title = entry.get("title", "")
    results = ner(title[:512])
    for r in results:
        if not is_valid_span(r["word"]):
            continue                          # skip garbage spans
        all_confidences.append(r["score"])
        if r["score"] < 0.70:
            flagged.append((r["word"], r["entity_group"], r["score"], title))

print(f"Total spans:      {len(all_confidences)}")
print(f"Mean confidence:  {sum(all_confidences)/len(all_confidences):.3f}")
print(f"Flagged (<0.70):  {len(flagged)}")
print("\nFlagged entities:")
for word, etype, score, title in flagged:
    print(f"  ⚠️  '{word}' ({etype}) {score:.3f} — {title[:60]}")

In [ ]:
def extract_entities(text):
    results = ner(text[:512])
    clean = []
    for r in results:
        word = r["word"]
        # Drop subword fragments
        if word.startswith("##"):
            continue
        # Drop single characters
        if len(word.strip("#")) <= 1:
            continue
        # Drop pure abbreviation fragments (LL, ##WS, etc.)
        if word.isupper() and len(word) <= 2:
            continue
        clean.append({
            "entity":     word,
            "type":       r["entity_group"],
            "confidence": round(r["score"], 4),
            "flagged":    r["score"] < 0.70,
        })
    return clean

In [ ]:
"""
pipeline/ner.py
================
Load raw articles from S3 → run NER → save entities + drift log to S3.

S3 inputs:  raw/YYYY-Www/batch_id.json
S3 outputs: entities/YYYY-Www/batch_id.json
            drift/YYYY-Www/batch_id.json
            label-queue/YYYY-Www/span_id.json  (low-confidence spans only)
"""

import datetime
from transformers import pipeline as hf_pipeline
from s3_utils import (
    read_all_json, write_json, append_drift_log, week_key, list_keys, read_json
)
from config import NER_MODEL, CONFIDENCE_THRESH

# Load model once at startup — not on every call
print(f"Loading NER model: {NER_MODEL}")
ner = hf_pipeline("ner", model=NER_MODEL, aggregation_strategy="simple")
print("Model loaded ✅")


def extract_entities(text: str) -> list[dict]:
    """Run NER on text, return list of entity dicts with confidence scores."""
    if not text or len(text.strip()) < 10:
        return []
    try:
        results = ner(text[:512])   # BERT max tokens
        return [
            {
                "entity":     r["word"],
                "type":       r["entity_group"],
                "confidence": round(r["score"], 4),
                "flagged":    r["score"] < CONFIDENCE_THRESH,
                "start":      r["start"],
                "end":        r["end"],
            }
            for r in results
        ]
    except Exception as e:
        print(f"  ⚠️  NER error: {e}")
        return []


def process_batch(batch: dict) -> dict:
    """Run NER on all articles in a batch, return entity results."""
    results = []
    all_confidences = []
    flagged_spans = []

    for article in batch.get("articles", []):
        text = f"{article['title']} {article.get('summary', '')}"
        entities = extract_entities(text)

        for ent in entities:
            all_confidences.append(ent["confidence"])
            if ent["flagged"]:
                flagged_spans.append({
                    "span_id":    f"{article['id']}_{ent['start']}",
                    "entity":     ent["entity"],
                    "type":       ent["type"],
                    "confidence": ent["confidence"],
                    "context":    text[:200],
                    "article_id": article["id"],
                    "week":       batch["week"],
                    "status":     "pending_label",
                })

        results.append({
            "article_id": article["id"],
            "title":      article["title"],
            "entities":   entities,
        })

    return {
        "batch_id":        batch["batch_id"],
        "week":            batch["week"],
        "article_count":   len(batch.get("articles", [])),
        "entity_results":  results,
        "all_confidences": all_confidences,
        "flagged_spans":   flagged_spans,
    }


def run():
    week = week_key()
    print(f"\n[NER] Processing week {week}")

    # Find unprocessed batches — raw/ items not yet in entities/
    raw_keys    = set(k.split("/")[-1] for k in list_keys(f"raw/{week}"))
    entity_keys = set(k.split("/")[-1] for k in list_keys(f"entities/{week}"))
    pending     = [k for k in raw_keys if k not in entity_keys]

    if not pending:
        print("  No new batches to process")
        return

    print(f"  {len(pending)} batches to process")

    for key_name in pending:
        batch_id = key_name.replace(".json", "")
        raw_data = read_json(f"raw/{week}/{key_name}")
        print(f"  Processing batch: {batch_id} ({len(raw_data.get('articles', []))} articles)")

        result = process_batch(raw_data)

        # Save entity results
        write_json(f"entities/{week}", batch_id, result)

        # Save drift log
        if result["all_confidences"]:
            append_drift_log(batch_id, result["all_confidences"], week)

        # Save each flagged span to label queue
        for span in result["flagged_spans"]:
            write_json(f"label-queue/{week}", span["span_id"], span)
            print(f"    ⚠️  Flagged: '{span['entity']}' (conf: {span['confidence']})")

        print(f"  ✅ {len(result['entity_results'])} articles processed, "
              f"{len(result['flagged_spans'])} spans flagged")


if __name__ == "__main__":
    run()


In [ ]:
# graph_explorer.ipynb cell 1 — install if needed
# !pip install pyvis plotly pandas networkx

In [ ]:
# cell 2 — imports and data loading
import json
import os
import glob
import datetime
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pyvis.network import Network
from IPython.display import HTML, display
import ipywidgets as widgets

# ── Config ──────────────────────────────────────────────────────────
LOCAL_DIR = "/tmp/pipeline_test"   # change to your local output dir

# ── Colour scheme ───────────────────────────────────────────────────
TYPE_COLORS = {
    "ORG":  "#1565c0",   # blue
    "MISC": "#2e7d32",   # green  (model names)
    "PER":  "#e65100",   # orange
    "LOC":  "#6a1b9a",   # purple
}

CONF_HIGH   = "#2e7d32"   # green  ≥ 0.85
CONF_MED    = "#f9a825"   # amber  0.70–0.85
CONF_LOW    = "#c62828"   # red    < 0.70

In [ ]:
# cell 3 — data loader
def load_all_entity_batches(base_dir=LOCAL_DIR):
    """Load all entity JSON files produced by ner.py."""
    pattern = os.path.join(base_dir, "entities", "**", "*.json")
    files   = sorted(glob.glob(pattern, recursive=True))

    batches = []
    for f in files:
        with open(f) as fh:
            data = json.load(fh)
        data["_file"]    = f
        data["_loaded"]  = datetime.datetime.fromtimestamp(os.path.getmtime(f))
        batches.append(data)
    return batches


def load_drift_logs(base_dir=LOCAL_DIR):
    """Load all drift log JSON files produced by drift.py."""
    pattern = os.path.join(base_dir, "drift", "**", "*.json")
    files   = sorted(glob.glob(pattern, recursive=True))
    logs = []
    for f in files:
        with open(f) as fh:
            data = json.load(fh)
        logs.append(data)
    return logs


def flatten_entities(batches):
    """Turn nested batch structure into a flat DataFrame."""
    rows = []
    for batch in batches:
        week     = batch.get("week", "unknown")
        batch_id = batch.get("batch_id", "unknown")
        loaded   = batch.get("_loaded", None)

        for article in batch.get("entity_results", []):
            title = article.get("title", "")
            for ent in article.get("entities", []):
                rows.append({
                    "entity":     ent["entity"],
                    "type":       ent["type"],
                    "confidence": ent["confidence"],
                    "flagged":    ent.get("flagged", ent["confidence"] < 0.70),
                    "article":    title,
                    "batch_id":   batch_id,
                    "week":       week,
                    "logged_at":  loaded,
                })
    return pd.DataFrame(rows)


# ── If no local files yet, generate mock data for testing ────────────
def mock_data():
    """Simulated pipeline output so you can build the UI before the pipeline runs."""
    import random, numpy as np
    random.seed(42)

    entities_raw = [
        # (name, type, base_conf, week)
        ("OpenAI",        "ORG",  0.996, "2025-W10"),
        ("Anthropic",     "ORG",  0.991, "2025-W10"),
        ("Google",        "ORG",  0.997, "2025-W10"),
        ("GPT-5",         "MISC", 0.983, "2025-W10"),
        ("Claude",        "MISC", 0.978, "2025-W10"),
        ("Gemini",        "MISC", 0.961, "2025-W10"),
        ("Sam Altman",    "PER",  0.988, "2025-W10"),
        ("Dario Amodei",  "PER",  0.972, "2025-W10"),
        # New entities — OOV, lower confidence
        ("DeepSeek",      "ORG",  0.910, "2025-W11"),
        ("DeepSeek R2",   "MISC", 0.620, "2025-W11"),
        ("xAI",           "ORG",  0.986, "2025-W11"),
        ("Grok 3",        "MISC", 0.576, "2025-W11"),
        ("SSI",           "ORG",  0.540, "2025-W12"),
        ("Siaivo",        "MISC", 0.490, "2025-W12"),
        ("Ilya Sutskever","PER",  0.960, "2025-W12"),
        ("Mistral AI",    "ORG",  0.870, "2025-W12"),
    ]

    co_occurrences = [
        ("OpenAI",    "GPT-5",      "INTRODUCES",  "2025-W10"),
        ("Anthropic", "Claude",     "INTRODUCES",  "2025-W10"),
        ("Google",    "Gemini",     "INTRODUCES",  "2025-W10"),
        ("GPT-5",     "Claude",     "COMPETES",    "2025-W10"),
        ("OpenAI",    "Anthropic",  "COMPETES",    "2025-W10"),
        ("DeepSeek",  "DeepSeek R2","INTRODUCES",  "2025-W11"),
        ("DeepSeek R2","GPT-5",     "OUTPERFORMS", "2025-W11"),
        ("xAI",       "Grok 3",     "INTRODUCES",  "2025-W11"),
        ("Grok 3",    "Claude",     "COMPETES",    "2025-W11"),
        ("SSI",       "Ilya Sutskever","BUILT_BY",  "2025-W12"),
        ("Mistral AI","OpenAI",     "COMPETES",    "2025-W12"),
    ]

    rows = []
    base_time = datetime.datetime(2025, 3, 1)
    for name, etype, base_conf, week in entities_raw:
        # Add some variation across articles
        for i in range(random.randint(2, 8)):
            jitter = random.gauss(0, 0.02)
            conf   = max(0.1, min(0.999, base_conf + jitter))
            wk_num = int(week.split("W")[1])
            rows.append({
                "entity":     name,
                "type":       etype,
                "confidence": round(conf, 4),
                "flagged":    conf < 0.70,
                "article":    f"Article about {name} — week {wk_num}",
                "batch_id":   f"{week}_mock",
                "week":       week,
                "logged_at":  base_time + datetime.timedelta(weeks=wk_num - 10, hours=i),
            })

    df = pd.DataFrame(rows)

    # Build edge list
    edges = pd.DataFrame(co_occurrences,
                         columns=["source", "target", "relation", "week"])
    return df, edges


# ── Load data ────────────────────────────────────────────────────────
batches = load_all_entity_batches()

if batches:
    df    = flatten_entities(batches)
    edges = pd.DataFrame(columns=["source","target","relation","week"])
    print(f"Loaded {len(batches)} batches — {len(df)} entity records")
else:
    print("No local pipeline data found — using mock data")
    df, edges = mock_data()

print(f"Last logged: {df['logged_at'].max()}")
print(f"Weeks:       {sorted(df['week'].unique())}")

In [ ]:
# cell 4 — confidence table view
def show_confidence_table(df, week_filter=None, min_conf=0.0, max_conf=1.0):
    filtered = df.copy()
    if week_filter:
        filtered = filtered[filtered["week"] == week_filter]
    filtered = filtered[
        (filtered["confidence"] >= min_conf) &
        (filtered["confidence"] <= max_conf)
    ]

    # Aggregate: mean conf, mention count, first seen, flagged count
    agg = (filtered.groupby(["entity", "type"])
           .agg(
               mean_conf   = ("confidence", "mean"),
               mentions    = ("entity",     "count"),
               first_seen  = ("week",       "min"),
               last_seen   = ("week",       "max"),
               flagged_n   = ("flagged",    "sum"),
               last_logged = ("logged_at",  "max"),
           )
           .reset_index()
           .sort_values("mentions", ascending=False))

    agg["mean_conf"]   = agg["mean_conf"].round(3)
    agg["status"]      = agg["mean_conf"].apply(
        lambda c: "🔴 LOW" if c < 0.70 else ("🟡 MED" if c < 0.85 else "🟢 OK"))

    # Plotly table
    fig = go.Figure(data=[go.Table(
        columnwidth=[120, 60, 80, 60, 60, 60, 70, 120],
        header=dict(
            values=["<b>Entity</b>","<b>Type</b>","<b>Mean Conf</b>",
                    "<b>Mentions</b>","<b>First Seen</b>","<b>Last Seen</b>",
                    "<b>Flagged</b>","<b>Last Logged</b>"],
            fill_color="#1565c0",
            font=dict(color="white", size=12),
            align="left",
        ),
        cells=dict(
            values=[
                agg["entity"],
                agg["type"],
                agg["mean_conf"],
                agg["mentions"],
                agg["first_seen"],
                agg["last_seen"],
                agg["flagged_n"].astype(str) + " / " + agg["mentions"].astype(str),
                agg["last_logged"].dt.strftime("%Y-%m-%d %H:%M").fillna("—"),
            ],
            fill_color=[
                ["#f8f9fa" if i % 2 == 0 else "white" for i in range(len(agg))],
            ],
            font=dict(size=11),
            align="left",
        )
    )])

    fig.update_layout(
        title=f"Entity Confidence Table — {week_filter or 'All Weeks'}",
        height=max(400, len(agg) * 28 + 80),
        margin=dict(l=20, r=20, t=60, b=20),
    )
    fig.show()

# ── Interactive widget wrapper ───────────────────────────────────────
week_options = ["All"] + sorted(df["week"].unique().tolist())

week_dd = widgets.Dropdown(options=week_options, value="All",
                           description="Week:")
conf_slider = widgets.FloatRangeSlider(value=[0.0, 1.0], min=0.0, max=1.0,
                                       step=0.01, description="Conf range:",
                                       layout=widgets.Layout(width="400px"))

def _update_table(week, conf_range):
    w = None if week == "All" else week
    show_confidence_table(df, week_filter=w,
                          min_conf=conf_range[0], max_conf=conf_range[1])

widgets.interactive(_update_table, week=week_dd, conf_range=conf_slider)

In [ ]:
# cell 5 — confidence timeline
def show_confidence_timeline(df):
    weekly = (df.groupby(["week", "type"])
              .agg(mean_conf=("confidence", "mean"),
                   flagged_pct=("flagged", "mean"))
              .reset_index())

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=("Mean Confidence by Entity Type per Week",
                        "Flagged Span % per Week (drift signal)"),
        shared_xaxes=True,
        vertical_spacing=0.12,
    )

    for etype, color in TYPE_COLORS.items():
        subset = weekly[weekly["type"] == etype]
        if subset.empty:
            continue
        fig.add_trace(go.Scatter(
            x=subset["week"], y=subset["mean_conf"],
            name=etype, line=dict(color=color, width=2),
            mode="lines+markers", marker=dict(size=7),
        ), row=1, col=1)

    # Flagged % — aggregated across all types
    flag_weekly = (df.groupby("week")
                   .agg(flagged_pct=("flagged", "mean"))
                   .reset_index())
    fig.add_trace(go.Bar(
        x=flag_weekly["week"], y=flag_weekly["flagged_pct"],
        name="Flagged %", marker_color="#e65100", opacity=0.7,
    ), row=2, col=1)

    # Drift threshold line
    fig.add_hline(y=0.72, row=1, col=1,
                  line=dict(color="red", dash="dash", width=1.5),
                  annotation_text="Drift threshold (0.72)")
    fig.add_hline(y=0.30, row=2, col=1,
                  line=dict(color="red", dash="dash", width=1.5),
                  annotation_text="Flag threshold (30%)")

    fig.update_layout(
        height=600, title="Confidence Trend — Drift Monitor",
        legend=dict(orientation="h", y=1.05),
        plot_bgcolor="#f8f9fa",
    )
    fig.update_yaxes(range=[0, 1.05], row=1, col=1)
    fig.update_yaxes(range=[0, 1.05], tickformat=".0%", row=2, col=1)
    fig.show()

show_confidence_timeline(df)

In [ ]:
# cell 6 — interactive knowledge graph (pyvis)
def build_graph(df, edges, week_filter=None):
    filtered_df    = df if week_filter is None else df[df["week"] <= week_filter]
    filtered_edges = edges if week_filter is None else edges[edges["week"] <= week_filter]

    # Aggregate node stats
    node_stats = (filtered_df.groupby(["entity","type"])
                  .agg(mentions=("entity","count"),
                       mean_conf=("confidence","mean"),
                       first_week=("week","min"))
                  .reset_index())

    G = nx.MultiDiGraph()

    for _, row in node_stats.iterrows():
        G.add_node(row["entity"],
                   etype=row["type"],
                   mentions=int(row["mentions"]),
                   conf=round(row["mean_conf"], 3),
                   first_week=row["first_week"])

    for _, row in filtered_edges.iterrows():
        if row["source"] in G.nodes and row["target"] in G.nodes:
            G.add_edge(row["source"], row["target"], relation=row["relation"])

    return G, node_stats


def show_knowledge_graph(df, edges, week_filter=None, height="700px"):
    G, node_stats = build_graph(df, edges, week_filter)

    net = Network(height=height, width="100%", bgcolor="#0f172a",
                  font_color="white", directed=True)

    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -80,
          "springLength": 120,
          "springConstant": 0.05
        },
        "solver": "forceAtlas2Based",
        "stabilization": { "iterations": 150 }
      },
      "edges": {
        "smooth": { "type": "curvedCW", "roundness": 0.2 },
        "arrows": { "to": { "enabled": true, "scaleFactor": 0.6 } },
        "font": { "size": 9, "color": "#94a3b8" }
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 100
      }
    }
    """)

    # Node size = mention count, opacity = confidence, colour = type
    max_mentions = max(nx.get_node_attributes(G, "mentions").values(), default=1)

    for node, attrs in G.nodes(data=True):
        etype      = attrs.get("etype", "ORG")
        conf       = attrs.get("conf", 0.9)
        mentions   = attrs.get("mentions", 1)
        first_week = attrs.get("first_week", "")
        week_filter_val = week_filter or ""

        base_color = TYPE_COLORS.get(etype, "#546e7a")
        size       = 12 + (mentions / max_mentions) * 30

        # Border: yellow if new entity (appeared in last week), else dark
        is_new    = first_week == week_filter_val
        border_c  = "#ffd600" if is_new else "#334155"
        border_w  = 3 if is_new else 1

        # Opacity via rgba — dim = low confidence
        opacity = max(0.3, conf)

        tooltip = (f"<b>{node}</b><br>"
                   f"Type: {etype}<br>"
                   f"Confidence: {conf:.3f}<br>"
                   f"Mentions: {mentions}<br>"
                   f"First seen: {first_week}")

        net.add_node(
            node,
            label=node,
            size=size,
            color={
                "background": base_color,
                "border":     border_c,
                "highlight":  {"background": "#fff", "border": "#ffd600"},
            },
            borderWidth=border_w,
            opacity=opacity,
            title=tooltip,
            font={"size": 11, "color": "white"},
        )

    # Edge colours by relation type
    EDGE_COLORS = {
        "INTRODUCES":   "#2e7d32",
        "OUTPERFORMS":  "#c62828",
        "COMPETES":     "#e65100",
        "PARTNERS":     "#1565c0",
        "FUNDS":        "#f9a825",
        "BUILT_BY":     "#6a1b9a",
    }

    for src, tgt, attrs in G.edges(data=True):
        rel   = attrs.get("relation", "RELATED")
        color = EDGE_COLORS.get(rel, "#64748b")
        net.add_edge(src, tgt, label=rel, color=color, width=1.5)

    # Legend as a disconnected node cluster (pyvis workaround)
    legend_x = -600
    for i, (etype, color) in enumerate(TYPE_COLORS.items()):
        net.add_node(f"_legend_{etype}", label=etype,
                     color={"background": color, "border": color},
                     size=10, x=legend_x, y=-200 + i * 50,
                     fixed=True, physics=False,
                     font={"size": 10, "color": "white"})

    html_path = "kg_graph.html"
    net.save_graph(html_path)
    display(HTML(filename=html_path))


# ── Week slider ──────────────────────────────────────────────────────
week_list   = sorted(df["week"].unique().tolist())
week_slider = widgets.SelectionSlider(
    options=["All"] + week_list,
    value="All",
    description="Up to week:",
    layout=widgets.Layout(width="500px"),
)

def _update_graph(week):
    w = None if week == "All" else week
    show_knowledge_graph(df, edges, week_filter=w)

widgets.interactive(_update_graph, week=week_slider)

In [ ]:
# cell 7 — batch log viewer
def show_batch_log(df):
    log = (df.groupby(["batch_id", "week", "logged_at"])
           .agg(
               articles   = ("article",    "nunique"),
               entities   = ("entity",     "count"),
               mean_conf  = ("confidence", "mean"),
               flagged    = ("flagged",    "sum"),
           )
           .reset_index()
           .sort_values("logged_at", ascending=False))

    log["mean_conf"]  = log["mean_conf"].round(3)
    log["drift_risk"] = log["mean_conf"].apply(
        lambda c: "🔴 HIGH" if c < 0.70 else ("🟡 WATCH" if c < 0.80 else "🟢 OK"))
    log["logged_at"]  = log["logged_at"].dt.strftime("%Y-%m-%d %H:%M")

    fig = go.Figure(data=[go.Table(
        header=dict(
            values=["<b>Batch ID</b>","<b>Week</b>","<b>Logged At</b>",
                    "<b>Articles</b>","<b>Entities</b>",
                    "<b>Mean Conf</b>","<b>Flagged</b>","<b>Drift Risk</b>"],
            fill_color="#0f172a",
            font=dict(color="white", size=12),
            align="left",
        ),
        cells=dict(
            values=[log["batch_id"], log["week"], log["logged_at"],
                    log["articles"], log["entities"],
                    log["mean_conf"], log["flagged"], log["drift_risk"]],
            fill_color=[["#1e293b" if i % 2 == 0 else "#0f172a"
                          for i in range(len(log))]],
            font=dict(color="white", size=11),
            align="left",
        )
    )])

    fig.update_layout(
        title="Pipeline Batch Log",
        height=max(300, len(log) * 30 + 80),
        paper_bgcolor="#0f172a",
        font_color="white",
        margin=dict(l=20, r=20, t=60, b=20),
    )
    fig.show()

show_batch_log(df)